# Video Swin Transformer — minimal implementation

**Paper:** Liu et al., *Video Swin Transformer*, CVPR 2022 — https://arxiv.org/abs/2106.13230
**Official code:** https://github.com/SwinTransformer/Video-Swin-Transformer
**Cleaner code (README link):** https://github.com/manmeet3591/video-swin-transformer-pytorch

Video Swin extends Swin to 3D: the input `(B, C, T, H, W)` is split into 3D patches, and self-attention is
computed inside local **3D windows** `(Wt, Wh, Ww)`. Every second block **shifts** the windows by half a window
so information flows between neighbouring windows. **Patch merging** halves H and W between stages (T is kept),
giving a hierarchical feature pyramid.

This notebook is a compact, dependency-light version (only `torch`), sized for a quick CPU/GPU smoke test.

In [1]:
from functools import reduce
from operator import mul

import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

## 3D window helpers

In [2]:
def window_partition(x, ws):
    """(B, D, H, W, C) -> (B*nW, Wd*Wh*Ww, C)"""
    B, D, H, W, C = x.shape
    x = x.view(B, D // ws[0], ws[0], H // ws[1], ws[1], W // ws[2], ws[2], C)
    return x.permute(0, 1, 3, 5, 2, 4, 6, 7).reshape(-1, reduce(mul, ws), C)


def window_reverse(windows, ws, B, D, H, W):
    """(B*nW, Wd, Wh, Ww, C) -> (B, D, H, W, C)"""
    x = windows.view(B, D // ws[0], H // ws[1], W // ws[2], ws[0], ws[1], ws[2], -1)
    return x.permute(0, 1, 4, 2, 5, 3, 6, 7).reshape(B, D, H, W, -1)


def get_window_size(x_size, window_size, shift_size):
    """Shrink the window (and drop the shift) on axes smaller than the window."""
    ws, ss = list(window_size), list(shift_size)
    for i in range(len(x_size)):
        if x_size[i] <= window_size[i]:
            ws[i], ss[i] = x_size[i], 0
    return tuple(ws), tuple(ss)


def compute_mask(D, H, W, ws, ss, device):
    """Attention mask so shifted windows don't attend across the wrap-around seam."""
    img_mask = torch.zeros((1, D, H, W, 1), device=device)
    cnt = 0
    for d in (slice(-ws[0]), slice(-ws[0], -ss[0]), slice(-ss[0], None)):
        for h in (slice(-ws[1]), slice(-ws[1], -ss[1]), slice(-ss[1], None)):
            for w in (slice(-ws[2]), slice(-ws[2], -ss[2]), slice(-ss[2], None)):
                img_mask[:, d, h, w, :] = cnt
                cnt += 1
    mask_windows = window_partition(img_mask, ws).squeeze(-1)
    attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
    return attn_mask.masked_fill(attn_mask != 0, -100.0).masked_fill(attn_mask == 0, 0.0)

## Window attention with 3D relative position bias

In [3]:
class WindowAttention3D(nn.Module):
    def __init__(self, dim, window_size, num_heads, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.window_size, self.num_heads = window_size, num_heads
        self.scale = (dim // num_heads) ** -0.5
        Wd, Wh, Ww = window_size
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * Wd - 1) * (2 * Wh - 1) * (2 * Ww - 1), num_heads))
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

        coords = torch.stack(torch.meshgrid(torch.arange(Wd), torch.arange(Wh), torch.arange(Ww), indexing="ij"))
        coords = coords.flatten(1)
        rel = (coords[:, :, None] - coords[:, None, :]).permute(1, 2, 0).contiguous()
        rel[:, :, 0] += Wd - 1
        rel[:, :, 1] += Wh - 1
        rel[:, :, 2] += Ww - 1
        rel[:, :, 0] *= (2 * Wh - 1) * (2 * Ww - 1)
        rel[:, :, 1] *= 2 * Ww - 1
        self.register_buffer("relative_position_index", rel.sum(-1), persistent=False)

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop, self.proj_drop = nn.Dropout(attn_drop), nn.Dropout(proj_drop)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        q, k, v = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        attn = (q * self.scale) @ k.transpose(-2, -1)
        bias = self.relative_position_bias_table[self.relative_position_index[:N, :N].reshape(-1)]
        attn = attn + bias.reshape(N, N, -1).permute(2, 0, 1).unsqueeze(0)
        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
        attn = self.attn_drop(attn.softmax(dim=-1))
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        return self.proj_drop(self.proj(x))

## Swin block, patch merging, patch embedding

In [4]:
class SwinBlock3D(nn.Module):
    def __init__(self, dim, num_heads, window_size, shift_size, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        self.window_size, self.shift_size = window_size, shift_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention3D(dim, window_size, num_heads, proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden, dim), nn.Dropout(drop))

    def _attn(self, x):
        B, D, H, W, C = x.shape
        ws, ss = get_window_size((D, H, W), self.window_size, self.shift_size)
        x = self.norm1(x)
        # pad to multiples of the window size
        pd, pb, pr = (ws[0] - D % ws[0]) % ws[0], (ws[1] - H % ws[1]) % ws[1], (ws[2] - W % ws[2]) % ws[2]
        x = F.pad(x, (0, 0, 0, pr, 0, pb, 0, pd))
        _, Dp, Hp, Wp, _ = x.shape
        if any(s > 0 for s in ss):
            x = torch.roll(x, shifts=(-ss[0], -ss[1], -ss[2]), dims=(1, 2, 3))
            mask = compute_mask(Dp, Hp, Wp, ws, ss, x.device)
        else:
            mask = None
        windows = self.attn(window_partition(x, ws), mask=mask)
        x = window_reverse(windows.view(-1, *ws, C), ws, B, Dp, Hp, Wp)
        if any(s > 0 for s in ss):
            x = torch.roll(x, shifts=ss, dims=(1, 2, 3))
        return x[:, :D, :H, :W, :].contiguous()

    def forward(self, x):  # x: (B, D, H, W, C)
        x = x + self._attn(x)
        return x + self.mlp(self.norm2(x))


class PatchMerging(nn.Module):
    """Halve H and W, double channels (T is not downsampled, as in the paper)."""
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(4 * dim)
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)

    def forward(self, x):
        B, D, H, W, C = x.shape
        x = F.pad(x, (0, 0, 0, W % 2, 0, H % 2))
        x = torch.cat([x[:, :, 0::2, 0::2], x[:, :, 1::2, 0::2], x[:, :, 0::2, 1::2], x[:, :, 1::2, 1::2]], -1)
        return self.reduction(self.norm(x))


class PatchEmbed3D(nn.Module):
    def __init__(self, patch_size=(2, 4, 4), in_chans=3, embed_dim=96):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv3d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):  # (B, C, T, H, W)
        _, _, D, H, W = x.shape
        p = self.patch_size
        x = F.pad(x, (0, (p[2] - W % p[2]) % p[2], 0, (p[1] - H % p[1]) % p[1], 0, (p[0] - D % p[0]) % p[0]))
        x = self.proj(x)                            # (B, E, D', H', W')
        return self.norm(x.permute(0, 2, 3, 4, 1))  # channels-last

## Full model

`forward_features` returns the multi-scale feature maps (useful as a backbone for forecasting / segmentation
heads); `forward` adds a global-pool classification head as in the paper (Kinetics action recognition).

In [5]:
class VideoSwinTransformer(nn.Module):
    def __init__(self, in_chans=3, num_classes=400, patch_size=(2, 4, 4), embed_dim=96,
                 depths=(2, 2, 6, 2), num_heads=(3, 6, 12, 24), window_size=(8, 7, 7), mlp_ratio=4.0, drop=0.0):
        super().__init__()
        self.patch_embed = PatchEmbed3D(patch_size, in_chans, embed_dim)
        self.stages, self.downsamples = nn.ModuleList(), nn.ModuleList()
        dim = embed_dim
        shift = tuple(w // 2 for w in window_size)
        for i, (depth, heads) in enumerate(zip(depths, num_heads)):
            self.stages.append(nn.Sequential(*[
                SwinBlock3D(dim, heads, window_size, (0, 0, 0) if j % 2 == 0 else shift, mlp_ratio, drop)
                for j in range(depth)]))
            last = i == len(depths) - 1
            self.downsamples.append(nn.Identity() if last else PatchMerging(dim))
            dim = dim if last else dim * 2
        self.num_features = dim
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def forward_features(self, x):
        x = self.patch_embed(x)
        feats = []
        for stage, down in zip(self.stages, self.downsamples):
            x = stage(x)
            feats.append(x.permute(0, 4, 1, 2, 3))  # (B, C, D, H, W)
            x = down(x)
        return feats

    def forward(self, x):
        x = self.forward_features(x)[-1].permute(0, 2, 3, 4, 1)
        x = self.norm(x).mean(dim=(1, 2, 3))
        return self.head(x)

## Smoke test

Swin-T configuration (≈28M params) on a dummy clip of 16 frames at 112×112. The paper uses 32×224×224.

In [6]:
model = VideoSwinTransformer(num_classes=10).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

x = torch.randn(2, 3, 16, 112, 112, device=device)  # (B, C, T, H, W)
with torch.no_grad():
    logits = model(x)
    feats = model.forward_features(x)
print("logits:", tuple(logits.shape))
for i, f in enumerate(feats):
    print(f"stage {i}:", tuple(f.shape))

params: 27.9M


logits: (2, 10)
stage 0: (2, 96, 8, 28, 28)
stage 1: (2, 192, 8, 14, 14)
stage 2: (2, 384, 8, 7, 7)
stage 3: (2, 768, 8, 4, 4)


## One training step on dummy data

In [7]:
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
y = torch.randint(0, 10, (2,), device=device)
loss = F.cross_entropy(model(x), y)
loss.backward()
opt.step()
print("loss:", loss.item())

loss: 2.45259952545166


## Using it on non-video spatiotemporal data

Any `(B, C, T, H, W)` tensor works, e.g. 4 weather variables over 8 time steps on a 64×128 grid.
On small grids, use a smaller patch and window so the windows fit.

In [8]:
wx = VideoSwinTransformer(in_chans=4, num_classes=1, patch_size=(1, 4, 4), embed_dim=48,
                          depths=(2, 2), num_heads=(3, 6), window_size=(4, 4, 4)).to(device)
with torch.no_grad():
    for f in wx.forward_features(torch.randn(1, 4, 8, 64, 128, device=device)):
        print(tuple(f.shape))

(1, 48, 8, 16, 32)
(1, 96, 8, 8, 16)
